# The Thermodynamic and Economic Consequences of Terafab on the United States Energy and Water through 2030

This Google Colab notebook uses **`terafab-decision-twin`** as the modeling kernel and acts as the reproducible publication-analysis layer for deterministic scenario analysis, derived indicators, uncertainty propagation, reduced-order stress trajectories, validation-readiness screening, and manuscript-quality figure export.

**Rights and evidence boundary.** This repository is source-available, not open source. The analyses below use public examples and declared assumptions only. Outputs are conditional analytical estimates from `terafab-decision-twin`; they are not verified Terafab operating facts, engineering certifications, investment advice, permitting determinations, procurement guidance, public-policy endorsements, or evidence of Terafab affiliation, authorization, financing, construction, operation, or adoption.

**Publication boundary.** The figures are formatted for manuscript preparation in the style expected of energy-systems, thermodynamic, and techno-economic studies, but this notebook does not claim journal acceptance or external validation.

## 1. Colab setup, package installation, and imports

The cell below locates the repository root, installs the local package in editable mode when possible, imports `terafab-decision-twin`, and records the run environment. It also installs notebook-only plotting/data dependencies if they are absent.

In [ ]:
from __future__ import annotations

import copy
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


def ensure_packages(packages):
    missing = [pkg for pkg, mod in packages.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

ensure_packages({"pandas": "pandas", "matplotlib": "matplotlib", "numpy": "numpy"})

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents, Path('/content/terafab-decision-twin'), Path('/workspace/terafab-decision-twin')]:
    if (candidate / 'terafab_decision_twin').exists() and (candidate / 'pyproject.toml').exists():
        ROOT = candidate.resolve()
        break
else:
    raise RuntimeError('Could not locate terafab-decision-twin source. Clone or upload the repository before running this notebook.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Install the local source package when available. This is valid in Colab and local Jupyter.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(ROOT)])

from terafab_decision_twin.engine import MODEL_VERSION, run_scenario
from terafab_decision_twin.schema import load_scenario, validate_scenario
from strategic_simulation import run_monte_carlo, simulate_reduced_order_model
from validation_lab import validate_result_structure, validate_reference_ranges, validation_scorecard

env_df = pd.DataFrame([{
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'repository_root': str(ROOT),
    'terafab_model_version': MODEL_VERSION,
}])
display(env_df)

## 2. Research questions, model boundary, and figure map

Every figure below is mapped to a research question and to explicit `terafab-decision-twin` outputs. This prevents the notebook from becoming a dashboard without manuscript logic.

In [ ]:
research_map_df = pd.DataFrame([
    {
        'research_question': 'How does Terafab-scale electric load propagate into heat rejection, entropy generation, and exergy destruction through 2030?',
        'terafab_outputs': 'energy_MWh; heat_rejection_required_MW; entropy_generation_MW_per_K; exergy_destroyed_MW; exergy_efficiency',
        'figure_ids': 'Fig. 2; Fig. 3',
        'claim_boundary': 'Conditional thermodynamic model output from declared scenarios.',
        'validation_requirement': 'Scenario validation, result hashes, equation traceability.'
    },
    {
        'research_question': 'Where do power, cooling, water, and wastewater margins become binding?',
        'terafab_outputs': 'firm_capacity_margin_MW; heat_rejection_margin_MW; water_withdrawal_margin_m3_per_day; wastewater_discharge_margin_m3_per_day',
        'figure_ids': 'Fig. 4; Fig. 9; Fig. 12',
        'claim_boundary': 'Modeled due-diligence screen, not permitting approval.',
        'validation_requirement': 'Gate checks and zero-margin reference lines.'
    },
    {
        'research_question': 'How do modeled energy and water burdens co-vary through 2030?',
        'terafab_outputs': 'energy_MWh; water_withdrawal_m3; water_consumptive_use_m3; wastewater_m3',
        'figure_ids': 'Fig. 5',
        'claim_boundary': 'Scenario-conditioned water-energy nexus.',
        'validation_requirement': 'Explicit units and derived water-per-MWh indicators.'
    },
    {
        'research_question': 'How do thermodynamic and water burdens translate into economic consequences?',
        'terafab_outputs': 'annualized_capex_USD; total_opex_USD; total_cost_USD; cost_per_good_die_USD; cost_per_compute_watt_USD',
        'figure_ids': 'Fig. 6; Fig. 7; Fig. 8',
        'claim_boundary': 'Scenario-conditioned techno-economic output.',
        'validation_requirement': 'Yield/readiness denominator context.'
    },
    {
        'research_question': 'Which declared assumptions dominate cost, feasibility, and legitimacy uncertainty?',
        'terafab_outputs': 'run_monte_carlo metric quantiles; sensitivity; gate failure probability',
        'figure_ids': 'Fig. 10; Fig. 11; Fig. 12',
        'claim_boundary': 'Declared-input uncertainty, not empirical validation.',
        'validation_requirement': 'Distribution table and reproducible seed.'
    },
    {
        'research_question': 'Which evidence and validation gaps prevent stronger claims?',
        'terafab_outputs': 'evidence.status_counts; unknowns; validation_lab checks; reference range flags',
        'figure_ids': 'Fig. 14',
        'claim_boundary': 'Validation-readiness only.',
        'validation_requirement': 'Explicit unsupported-claims table.'
    },
])
display(research_map_df)

## 3. Publication plotting standard

The style helpers below enforce vector-first export, colorblind-aware palettes, unit-bearing labels, margin reference lines, manifest-ready captions, and consistent file output.

In [ ]:
FIG_DIR = ROOT / 'publication_figures'
FIG_DIR.mkdir(exist_ok=True)

SCENARIO_COLORS = {
    'Baseline': '#0072B2',
    'Best case': '#009E73',
    'Worst case': '#D55E00',
    'Multi-year planning case': '#CC79A7',
    'Terawatt stress case': '#000000',
}
LINE_STYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]
figure_manifest = []

def set_publication_style():
    plt.rcParams.update({
        'figure.dpi': 120,
        'savefig.dpi': 600,
        'font.size': 9,
        'axes.labelsize': 9,
        'axes.titlesize': 10,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
        'lines.linewidth': 1.8,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': True,
        'grid.alpha': 0.25,
        'grid.linewidth': 0.5,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    })


def save_publication_figure(fig, stem, title, primary_claim, primary_metrics, source_dataframe, scenarios_used, method_boundary, evidence_boundary):
    paths = {}
    for ext in ['svg', 'pdf', 'png']:
        path = FIG_DIR / f'{stem}.{ext}'
        fig.savefig(path, bbox_inches='tight', facecolor='white')
        paths[ext] = str(path.relative_to(ROOT))
    figure_manifest.append({
        'figure_id': stem,
        'figure_title': title,
        'filename_svg': paths['svg'],
        'filename_pdf': paths['pdf'],
        'filename_png': paths['png'],
        'primary_claim': primary_claim,
        'primary_metrics': primary_metrics,
        'source_dataframe': source_dataframe,
        'scenarios_used': scenarios_used,
        'method_boundary': method_boundary,
        'evidence_boundary': evidence_boundary,
        'suggested_caption': f'{title}. {primary_claim} {method_boundary} {evidence_boundary}',
    })
    return paths


def add_panel_label(ax, label):
    ax.text(-0.10, 1.06, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='top')


def safe_divide(a, b):
    if b is None or pd.isna(b) or float(b) == 0.0:
        return np.nan
    return float(a) / float(b)

set_publication_style()

## 4. Scenario comparator design and validation

The notebook uses `terafab-decision-twin` scenario JSON files as comparator cases. These are declared scenarios, not verified Terafab operating cases.

In [ ]:
scenario_specs = [
    ('Baseline', ROOT / 'scenarios/baseline_2026.json', 'Reference case'),
    ('Best case', ROOT / 'scenarios/best_case_2026_2030.json', 'Favorable infrastructure/readiness assumptions'),
    ('Worst case', ROOT / 'scenarios/worst_case_2026_2030.json', 'Adverse infrastructure/readiness assumptions'),
    ('Multi-year planning case', ROOT / 'scenarios/multi_year_2026_2030.json', 'Through-2030 planning trajectory'),
    ('Terawatt stress case', ROOT / 'scenarios/terawatt_stress_2026.json', 'Stress-test only; not a verified operating case'),
]

scenarios = {}
registry_rows = []
for label, path, boundary in scenario_specs:
    scenario = load_scenario(path)
    errors = validate_scenario(scenario)
    scenarios[label] = scenario
    meta = scenario.get('metadata', {})
    time = scenario.get('time', {})
    registry_rows.append({
        'scenario_label': label,
        'scenario_id': meta.get('scenario_id'),
        'scenario_title': meta.get('title'),
        'scenario_type': meta.get('scenario_type'),
        'start_year': time.get('start_year'),
        'end_year': time.get('end_year'),
        'time_step': time.get('step'),
        'file_path': str(path.relative_to(ROOT)),
        'validation_passed': not errors,
        'validation_errors': '; '.join(errors),
        'claim_boundary': boundary,
    })
scenario_registry_df = pd.DataFrame(registry_rows)
display(scenario_registry_df)
assert scenario_registry_df['validation_passed'].all(), 'One or more scenarios failed schema validation.'

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 2.8))
plot_df = scenario_registry_df.copy()
y = np.arange(len(plot_df))
starts = plot_df['start_year'].astype(float)
ends = plot_df['end_year'].astype(float)
ax.barh(y, ends - starts + 1, left=starts, color=[SCENARIO_COLORS.get(s, '#666666') for s in plot_df['scenario_label']], alpha=0.8)
ax.set_yticks(y)
ax.set_yticklabels(plot_df['scenario_label'])
ax.set_xlabel('Modeled year')
ax.set_title('Figure 1. Scenario comparator design')
ax.invert_yaxis()
for idx, row in plot_df.iterrows():
    ax.text(float(row['end_year']) + 0.05, idx, row['claim_boundary'], va='center', fontsize=7)
fig.tight_layout()
save_publication_figure(fig, 'fig_01_comparator_design', 'Scenario comparator design', 'The analysis uses declared comparator scenarios spanning baseline, favorable, adverse, planning, and stress-test cases.', 'scenario horizon; scenario type', 'scenario_registry_df', ', '.join(plot_df['scenario_label']), 'Scenario JSON files are loaded and schema validated before execution.', 'Comparator cases are declared scenario inputs, not verified operating facts.')
plt.show()

## 5. Deterministic execution with `terafab-decision-twin`

The deterministic kernel is `run_scenario()` from `terafab-decision-twin`. The notebook reshapes outputs into tidy dataframes for analysis and plotting.

In [ ]:
results_by_label = {label: run_scenario(scenario) for label, scenario in scenarios.items()}

summary_rows, ts_rows, gate_rows, evidence_rows, hash_rows, output_record_rows = [], [], [], [], [], []
for label, result in results_by_label.items():
    sid = result['metadata']['scenario_id']
    row = dict(result['summary'])
    row.update({'scenario_label': label, 'scenario_id': sid, 'passed': result.get('passed'), 'model_version': result['metadata'].get('version')})
    summary_rows.append(row)
    for t in result['time_series']:
        r = dict(t); r.update({'scenario_label': label, 'scenario_id': sid}); ts_rows.append(r)
    for gate in result['gates']:
        r = dict(gate); r.update({'scenario_label': label, 'scenario_id': sid}); gate_rows.append(r)
    for status, count in result['evidence']['status_counts'].items():
        evidence_rows.append({'scenario_label': label, 'scenario_id': sid, 'evidence_status': status, 'count': count})
    hashes = dict(result['hashes']); hashes.update({'scenario_label': label, 'scenario_id': sid}); hash_rows.append(hashes)
    for rec in result['output_records']:
        r = dict(rec); r.update({'scenario_label': label}); output_record_rows.append(r)

summary_df = pd.DataFrame(summary_rows)
timeseries_df = pd.DataFrame(ts_rows)
gates_df = pd.DataFrame(gate_rows)
evidence_df = pd.DataFrame(evidence_rows)
hashes_df = pd.DataFrame(hash_rows)
output_records_df = pd.DataFrame(output_record_rows)

compact_columns = [c for c in [
    'scenario_label','scenario_id','energy_MWh','heat_rejection_required_MW','entropy_generation_MW_per_K','exergy_destroyed_MW','average_exergy_efficiency',
    'water_withdrawal_m3','water_consumptive_use_m3','wastewater_m3','minimum_firm_capacity_margin_MW','minimum_heat_rejection_margin_MW',
    'minimum_water_withdrawal_margin_m3_per_day','minimum_wastewater_discharge_margin_m3_per_day','annualized_capex_USD','total_opex_USD','total_cost_USD',
    'cost_per_good_die_USD','cost_per_compute_watt_USD','emissions_tCO2','legitimacy_margin','recommended_phase_action','passed'
] if c in summary_df.columns]
display(summary_df[compact_columns])
display(hashes_df)

## 6. Derived normalized and dimensionless indicators

Raw totals are supplemented with normalized indicators so scenario comparisons are interpretable across different horizons and magnitudes.

In [ ]:
derived_metric_definitions_df = pd.DataFrame([
    ('heat_rejection_to_load_ratio', 'heat_rejection_required_MW / site_electric_load_MW', 'ratio'),
    ('exergy_destroyed_per_MWh', 'exergy_destroyed_MW / energy_MWh', 'MW/MWh'),
    ('cooling_auxiliary_energy_share', 'cooling_auxiliary_energy_MWh / energy_MWh', 'fraction'),
    ('exergy_efficiency_percent', 'exergy_efficiency * 100', 'percent'),
    ('water_withdrawal_per_MWh', 'water_withdrawal_m3 / energy_MWh', 'm3/MWh'),
    ('consumptive_use_per_MWh', 'water_consumptive_use_m3 / energy_MWh', 'm3/MWh'),
    ('wastewater_per_MWh', 'wastewater_m3 / energy_MWh', 'm3/MWh'),
    ('consumptive_use_share', 'water_consumptive_use_m3 / water_withdrawal_m3', 'fraction'),
    ('wastewater_share', 'wastewater_m3 / water_withdrawal_m3', 'fraction'),
    ('total_cost_per_MWh', 'total_cost_USD / energy_MWh', 'USD/MWh'),
    ('opex_per_MWh', 'total_opex_USD / energy_MWh', 'USD/MWh'),
], columns=['metric', 'formula', 'unit'])

derived_timeseries_df = timeseries_df.copy()
derived_timeseries_df['heat_rejection_to_load_ratio'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('heat_rejection_required_MW'), r.get('site_electric_load_MW')), axis=1)
derived_timeseries_df['exergy_destroyed_per_MWh'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('exergy_destroyed_MW'), r.get('energy_MWh')), axis=1)
derived_timeseries_df['cooling_auxiliary_energy_share'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('cooling_auxiliary_energy_MWh'), r.get('energy_MWh')), axis=1)
derived_timeseries_df['exergy_efficiency_percent'] = derived_timeseries_df['exergy_efficiency'] * 100.0
derived_timeseries_df['water_withdrawal_per_MWh'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('water_withdrawal_m3'), r.get('energy_MWh')), axis=1)
derived_timeseries_df['consumptive_use_per_MWh'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('water_consumptive_use_m3'), r.get('energy_MWh')), axis=1)
derived_timeseries_df['wastewater_per_MWh'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('wastewater_m3'), r.get('energy_MWh')), axis=1)
derived_timeseries_df['consumptive_use_share'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('water_consumptive_use_m3'), r.get('water_withdrawal_m3')), axis=1)
derived_timeseries_df['wastewater_share'] = derived_timeseries_df.apply(lambda r: safe_divide(r.get('wastewater_m3'), r.get('water_withdrawal_m3')), axis=1)

derived_summary_df = summary_df.copy()
derived_summary_df['total_cost_per_MWh'] = derived_summary_df.apply(lambda r: safe_divide(r.get('total_cost_USD'), r.get('energy_MWh')), axis=1)
derived_summary_df['opex_per_MWh'] = derived_summary_df.apply(lambda r: safe_divide(r.get('total_opex_USD'), r.get('energy_MWh')), axis=1)
derived_summary_df['cost_per_exergy_destroyed'] = derived_summary_df.apply(lambda r: safe_divide(r.get('total_cost_USD'), r.get('exergy_destroyed_MW')), axis=1)
derived_summary_df['exergy_efficiency_percent'] = derived_summary_df['average_exergy_efficiency'] * 100.0

display(derived_metric_definitions_df)
display(derived_summary_df[['scenario_label','total_cost_per_MWh','opex_per_MWh','cost_per_exergy_destroyed','exergy_efficiency_percent']])

## 7. Uncertainty design and declared input distributions

Uncertainty is a first-class result. The distributions below are declared modeling choices for stress testing; they are not empirical calibration distributions.

In [ ]:
MC_RUNS_COLAB = int(os.environ.get('TERAFAB_MC_RUNS', '80'))
MC_RUNS_FINAL_SUGGESTED = 2000
mc_parameters = {
    'energy.site_electric_load_MW': {'distribution': 'triangular', 'low': 120, 'mode': 150, 'high': 190, 'unit': 'MW', 'status': 'scenario_assumption'},
    'energy.firm_capacity_MW': {'distribution': 'triangular', 'low': 170, 'mode': 200, 'high': 240, 'unit': 'MW', 'status': 'scenario_assumption'},
    'cooling.heat_rejection_capacity_MW': {'distribution': 'triangular', 'low': 140, 'mode': 170, 'high': 220, 'unit': 'MW', 'status': 'scenario_assumption'},
    'cooling.cop': {'distribution': 'triangular', 'low': 4.5, 'mode': 6.0, 'high': 7.5, 'unit': 'ratio', 'status': 'scenario_assumption'},
    'water.withdrawal_m3_per_MWh': {'distribution': 'triangular', 'low': 0.8, 'mode': 1.2, 'high': 1.8, 'unit': 'm3/MWh', 'status': 'scenario_assumption'},
    'water.permit_withdrawal_m3_per_day': {'distribution': 'triangular', 'low': 3500, 'mode': 5000, 'high': 6500, 'unit': 'm3/day', 'status': 'scenario_assumption'},
    'water.permit_discharge_m3_per_day': {'distribution': 'triangular', 'low': 1800, 'mode': 2500, 'high': 3500, 'unit': 'm3/day', 'status': 'scenario_assumption'},
    'manufacturing.baseline_yield': {'distribution': 'triangular', 'low': 0.55, 'mode': 0.72, 'high': 0.85, 'unit': 'fraction', 'status': 'scenario_assumption'},
    'manufacturing.learning_rate_per_year': {'distribution': 'triangular', 'low': 0.0, 'mode': 0.025, 'high': 0.06, 'unit': 'fraction/year', 'status': 'scenario_assumption'},
    'economics.electricity_price_USD_per_MWh': {'distribution': 'triangular', 'low': 45, 'mode': 75, 'high': 120, 'unit': 'USD/MWh', 'status': 'scenario_assumption'},
    'economics.capex_USD': {'distribution': 'triangular', 'low': 8e9, 'mode': 12e9, 'high': 18e9, 'unit': 'USD', 'status': 'scenario_assumption'},
    'policy.local_water_stress_index': {'distribution': 'triangular', 'low': 0.1, 'mode': 0.3, 'high': 0.7, 'unit': 'index', 'status': 'scenario_assumption'},
}
uncertainty_distribution_df = pd.DataFrame([
    {'dotted_path': path, **spec, 'reason_for_uncertainty': 'Material driver of load, feasibility, cost, water burden, or legitimacy.', 'claim_boundary': 'Declared input uncertainty only.'}
    for path, spec in mc_parameters.items()
])
display(uncertainty_distribution_df)

mc_config = {
    'runs': MC_RUNS_COLAB,
    'seed': 2030,
    'parameters': mc_parameters,
    'metrics': [
        'minimum_firm_capacity_margin_MW','minimum_heat_rejection_margin_MW','minimum_water_withdrawal_margin_m3_per_day',
        'minimum_wastewater_discharge_margin_m3_per_day','total_cost_USD','cost_per_good_die_USD','emissions_tCO2','legitimacy_margin'
    ],
    'sensitivity_target': 'total_cost_USD',
}
mc_result = run_monte_carlo(scenarios['Multi-year planning case'], mc_config, retain_runs=True)
monte_carlo_quantiles_df = pd.DataFrame([
    {'metric': metric, **vals} for metric, vals in mc_result['metric_quantiles'].items()
])
monte_carlo_sensitivity_df = pd.DataFrame(mc_result['sensitivity'])
gate_failure_probability_df = pd.DataFrame([
    {'gate': gate, 'failure_probability': probability} for gate, probability in mc_result['gate_failure_probability'].items()
])
display(monte_carlo_quantiles_df)
display(monte_carlo_sensitivity_df.head(10))
display(gate_failure_probability_df)

## 8. Thermodynamic consequences

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.2), sharex=True)
metrics = [('energy_MWh','Energy (MWh)'), ('heat_rejection_required_MW','Heat rejection required (MW)'), ('cooling_auxiliary_energy_MWh','Cooling auxiliary energy (MWh)')]
for ax, (metric, ylabel), panel in zip(axes, metrics, ['(a)','(b)','(c)']):
    for i, (label, grp) in enumerate(timeseries_df.groupby('scenario_label')):
        ax.plot(grp['year'], grp[metric], label=label, color=SCENARIO_COLORS.get(label), linestyle=LINE_STYLES[i % len(LINE_STYLES)], marker='o', markersize=3)
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].set_xlabel('Year')
axes[0].legend(ncol=2, frameon=False)
fig.suptitle('Figure 2. Energy demand and heat-rejection trajectory', y=0.995)
fig.tight_layout()
save_publication_figure(fig, 'fig_02_energy_heat_trajectory', 'Energy demand and heat-rejection trajectory', 'Modeled electric energy, heat rejection, and cooling auxiliary energy define the thermodynamic burden through the scenario horizon.', 'energy_MWh; heat_rejection_required_MW; cooling_auxiliary_energy_MWh', 'timeseries_df', ', '.join(timeseries_df['scenario_label'].unique()), 'Deterministic outputs from terafab-decision-twin run_scenario().', 'Conditional on declared scenario assumptions.')
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.2), sharex=True)
metrics = [('entropy_generation_MW_per_K','Entropy generation (MW/K)'), ('exergy_destroyed_MW','Exergy destroyed (MW)'), ('exergy_efficiency','Exergy efficiency (fraction)')]
for ax, (metric, ylabel), panel in zip(axes, metrics, ['(a)','(b)','(c)']):
    for i, (label, grp) in enumerate(timeseries_df.groupby('scenario_label')):
        ax.plot(grp['year'], grp[metric], label=label, color=SCENARIO_COLORS.get(label), linestyle=LINE_STYLES[i % len(LINE_STYLES)], marker='o', markersize=3)
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].set_xlabel('Year')
axes[0].legend(ncol=2, frameon=False)
fig.suptitle('Figure 3. Entropy and exergy consequences', y=0.995)
fig.tight_layout()
save_publication_figure(fig, 'fig_03_entropy_exergy', 'Entropy and exergy consequences', 'Entropy generation and exergy destruction expose thermodynamic losses implied by the declared scenarios.', 'entropy_generation_MW_per_K; exergy_destroyed_MW; exergy_efficiency', 'timeseries_df', ', '.join(timeseries_df['scenario_label'].unique()), 'Deterministic thermodynamic outputs from terafab-decision-twin.', 'Model-derived indicators, not measured plant data.')
plt.show()

## 9. Feasibility margins, water-energy nexus, economics, gates, and readiness

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(7.2, 8.2), sharex=True)
metrics = [('firm_capacity_margin_MW','Firm power margin (MW)'), ('heat_rejection_margin_MW','Heat-rejection margin (MW)'), ('water_withdrawal_margin_m3_per_day','Withdrawal margin (m3/day)'), ('wastewater_discharge_margin_m3_per_day','Discharge margin (m3/day)')]
for ax, (metric, ylabel), panel in zip(axes, metrics, ['(a)','(b)','(c)','(d)']):
    for i, (label, grp) in enumerate(timeseries_df.groupby('scenario_label')):
        ax.plot(grp['year'], grp[metric], label=label, color=SCENARIO_COLORS.get(label), linestyle=LINE_STYLES[i % len(LINE_STYLES)], marker='o', markersize=3)
    ax.axhline(0, color='black', linewidth=1.0)
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].set_xlabel('Year')
axes[0].legend(ncol=2, frameon=False)
fig.suptitle('Figure 4. Feasibility margins', y=0.995)
fig.tight_layout()
save_publication_figure(fig, 'fig_04_feasibility_margins', 'Feasibility margins', 'Zero-margin lines identify modeled power, cooling, water, and wastewater feasibility boundaries.', 'firm_capacity_margin_MW; heat_rejection_margin_MW; water and wastewater margins', 'timeseries_df', ', '.join(timeseries_df['scenario_label'].unique()), 'Deterministic due-diligence screen from terafab-decision-twin.', 'Not a permitting or engineering certification.')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8))
for metric, ylabel in [('water_withdrawal_m3','Withdrawal (m3)'), ('water_consumptive_use_m3','Consumptive use (m3)'), ('wastewater_m3','Wastewater (m3)')]:
    ax = axes[['water_withdrawal_m3','water_consumptive_use_m3','wastewater_m3'].index(metric)]
    for i, (label, grp) in enumerate(timeseries_df.groupby('scenario_label')):
        ax.plot(grp['year'], grp[metric], label=label, color=SCENARIO_COLORS.get(label), linestyle=LINE_STYLES[i % len(LINE_STYLES)], marker='o', markersize=3)
    ax.set_title(ylabel); ax.set_xlabel('Year')
axes[0].set_ylabel('Water flow')
axes[0].legend(frameon=False, fontsize=6)
fig.suptitle('Figure 5. Water-energy nexus', y=1.02)
fig.tight_layout()
save_publication_figure(fig, 'fig_05_water_energy_nexus', 'Water-energy nexus', 'Modeled withdrawal, consumption, and wastewater are first-order consequences of energy and manufacturing assumptions.', 'water_withdrawal_m3; water_consumptive_use_m3; wastewater_m3', 'timeseries_df', ', '.join(timeseries_df['scenario_label'].unique()), 'Deterministic water outputs plus derived indicators.', 'Conditional scenario output, not reported facility withdrawal.')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))
sdf = summary_df.set_index('scenario_label')
(sdf[['annualized_capex_USD','total_opex_USD','total_cost_USD']] / 1e9).plot(kind='bar', ax=axes[0], color=['#56B4E9','#E69F00','#999999'])
axes[0].set_ylabel('Billion USD'); axes[0].set_title('Cost totals')
derived_summary_df.set_index('scenario_label')[['total_cost_per_MWh','opex_per_MWh']].plot(kind='bar', ax=axes[1], color=['#0072B2','#009E73'])
axes[1].set_ylabel('USD/MWh'); axes[1].set_title('Cost intensity')
sdf[['cost_per_good_die_USD','cost_per_compute_watt_USD']].plot(kind='bar', ax=axes[2], color=['#D55E00','#CC79A7'])
axes[2].set_ylabel('USD per unit'); axes[2].set_title('Manufacturing-normalized cost')
for ax in axes:
    ax.tick_params(axis='x', labelrotation=60)
    ax.legend(frameon=False, fontsize=6)
fig.suptitle('Figure 6. Economic consequences and cost intensity', y=1.03)
fig.tight_layout()
save_publication_figure(fig, 'fig_06_cost_structure', 'Economic consequences and cost intensity', 'Cost consequences depend on capex, operating burdens, energy use, and yield/readiness denominators.', 'annualized_capex_USD; total_opex_USD; total_cost_USD; cost intensities', 'summary_df; derived_summary_df', ', '.join(summary_df['scenario_label'].unique()), 'Deterministic techno-economic outputs from terafab-decision-twin.', 'Scenario-conditioned estimates, not investment advice.')
plt.show()

fig, ax = plt.subplots(figsize=(4.6, 3.4))
for label, grp in derived_summary_df.groupby('scenario_label'):
    ax.scatter(grp['exergy_destroyed_MW'], grp['total_cost_per_MWh'], s=55, label=label, color=SCENARIO_COLORS.get(label))
ax.set_xlabel('Average exergy destroyed (MW)')
ax.set_ylabel('Total cost intensity (USD/MWh)')
ax.legend(frameon=False, fontsize=7)
ax.set_title('Figure 7. Thermoeconomic map')
fig.tight_layout()
save_publication_figure(fig, 'fig_07_thermoeconomic_map', 'Thermoeconomic map', 'Scenario-level exergy destruction is compared with modeled cost intensity to reveal thermoeconomic coupling.', 'exergy_destroyed_MW; total_cost_per_MWh', 'derived_summary_df', ', '.join(summary_df['scenario_label'].unique()), 'Cross-scenario deterministic association.', 'Not empirical causal inference.')
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(7.2, 6.8), sharex=True)
for ax, metric, ylabel, panel in zip(axes, ['effective_yield','readiness_index','good_die'], ['Effective yield (fraction)','Readiness index (fraction)','Good die'], ['(a)','(b)','(c)']):
    for i, (label, grp) in enumerate(timeseries_df.groupby('scenario_label')):
        ax.plot(grp['year'], grp[metric], label=label, color=SCENARIO_COLORS.get(label), linestyle=LINE_STYLES[i % len(LINE_STYLES)], marker='o', markersize=3)
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].set_xlabel('Year')
axes[0].legend(ncol=2, frameon=False)
fig.suptitle('Figure 8. Yield, readiness, and output context', y=0.995)
fig.tight_layout()
save_publication_figure(fig, 'fig_08_yield_readiness_output', 'Yield, readiness, and output context', 'Manufacturing readiness and yield affect output denominators used in cost-intensity results.', 'effective_yield; readiness_index; good_die', 'timeseries_df', ', '.join(timeseries_df['scenario_label'].unique()), 'Deterministic manufacturing outputs from terafab-decision-twin.', 'Conditional on declared manufacturing assumptions.')
plt.show()

pivot = gates_df.pivot_table(index='name', columns='scenario_label', values='passed', aggfunc='min').astype(float)
fig, ax = plt.subplots(figsize=(7.2, max(3.0, 0.25 * len(pivot))))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for y in range(pivot.shape[0]):
    for x in range(pivot.shape[1]):
        ax.text(x, y, 'P' if pivot.iloc[y,x] else 'F', ha='center', va='center', fontsize=7)
ax.set_title('Figure 9. Gate failure heatmap')
fig.colorbar(im, ax=ax, label='Passed')
fig.tight_layout()
save_publication_figure(fig, 'fig_09_gate_failure_heatmap', 'Gate failure heatmap', 'Model gates identify constraints and evidence issues that block decision readiness.', 'gate pass/fail status', 'gates_df', ', '.join(summary_df['scenario_label'].unique()), 'Gate results from terafab-decision-twin.', 'Not external due diligence or regulatory approval.')
plt.show()

## 10. Monte Carlo uncertainty, sensitivity, and gate risk

In [ ]:
qdf = monte_carlo_quantiles_df.dropna(subset=['p10','p50','p90']).copy()
fig, ax = plt.subplots(figsize=(7.2, 3.8))
y = np.arange(len(qdf))
left_err = qdf['p50'] - qdf['p10']
right_err = qdf['p90'] - qdf['p50']
ax.errorbar(qdf['p50'], y, xerr=[left_err, right_err], fmt='o', color='#0072B2', ecolor='#999999', capsize=3)
ax.set_yticks(y); ax.set_yticklabels(qdf['metric'])
ax.set_xlabel('Metric value (native units)')
ax.set_title('Figure 10. Declared-uncertainty quantiles')
fig.tight_layout()
save_publication_figure(fig, 'fig_10_uncertainty_quantiles', 'Declared-uncertainty quantiles', 'Monte Carlo p10-p50-p90 intervals show how declared input uncertainty propagates into feasibility, cost, emissions, and legitimacy metrics.', 'metric quantiles', 'monte_carlo_quantiles_df', 'Multi-year planning case', 'Declared distributions propagated with strategic_simulation.run_monte_carlo().', 'Not empirical confidence intervals or validation.')
plt.show()

sdf2 = monte_carlo_sensitivity_df.head(10).sort_values('absolute_correlation')
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.barh(sdf2['parameter'], sdf2['absolute_correlation'], color='#D55E00')
ax.set_xlabel('Absolute Pearson correlation with total_cost_USD')
ax.set_title('Figure 11. Sensitivity tornado')
fig.tight_layout()
save_publication_figure(fig, 'fig_11_sensitivity_tornado', 'Sensitivity tornado', 'The sensitivity ranking identifies declared inputs most associated with modeled total cost in the Monte Carlo experiment.', 'absolute_correlation', 'monte_carlo_sensitivity_df', 'Multi-year planning case', 'Local correlation-based sensitivity under declared distributions.', 'Does not prove causal dominance outside the declared scenario space.')
plt.show()

gdf = gate_failure_probability_df.sort_values('failure_probability')
fig, ax = plt.subplots(figsize=(7.2, max(3.0, 0.25 * len(gdf))))
ax.barh(gdf['gate'], gdf['failure_probability'], color='#CC79A7')
ax.set_xlabel('Failure probability under declared uncertainty')
ax.set_title('Figure 12. Gate failure probability')
fig.tight_layout()
save_publication_figure(fig, 'fig_12_gate_failure_probability', 'Gate failure probability', 'Gate failure probabilities summarize how often due-diligence gates fail under declared input uncertainty.', 'gate_failure_probability', 'gate_failure_probability_df', 'Multi-year planning case', 'Monte Carlo propagation through deterministic terafab-decision-twin gates.', 'Scenario-conditioned risk screen, not real-world probability.')
plt.show()

## 11. Reduced-order system stress trajectory

In [ ]:
rom_config = {
    'state_variables': ['power_stress','heat_rejection_stress','water_stress','yield_maturity','policy_legitimacy'],
    'control_variables': ['capacity_buildout','cooling_investment','water_reuse_investment','qualification_effort'],
    'steps': 5,
    'x0': {'power_stress':0.35,'heat_rejection_stress':0.40,'water_stress':0.30,'yield_maturity':0.45,'policy_legitimacy':0.55},
    'A': [[0.82,0.03,0.02,0,0], [0.04,0.84,0.02,0,0], [0.02,0.02,0.86,0,0], [0,0,0,0.90,0.02], [0.02,0.02,0.05,0.03,0.88]],
    'B': [[-0.12,0,0,0], [0,-0.12,0,0], [0,0,-0.15,0], [0,0,0,0.10], [-0.02,-0.02,-0.04,0.03]],
    'c': {'power_stress':0.04,'heat_rejection_stress':0.04,'water_stress':0.03,'yield_maturity':0.02,'policy_legitimacy':-0.01},
    'control_policy': [
        {'capacity_buildout':0.2,'cooling_investment':0.2,'water_reuse_investment':0.15,'qualification_effort':0.2},
        {'capacity_buildout':0.3,'cooling_investment':0.25,'water_reuse_investment':0.2,'qualification_effort':0.25},
        {'capacity_buildout':0.35,'cooling_investment':0.3,'water_reuse_investment':0.25,'qualification_effort':0.3},
        {'capacity_buildout':0.4,'cooling_investment':0.35,'water_reuse_investment':0.3,'qualification_effort':0.35},
        {'capacity_buildout':0.45,'cooling_investment':0.4,'water_reuse_investment':0.35,'qualification_effort':0.4},
    ],
    'clip_lower': 0.0,
    'clip_upper': 1.0,
}
rom_result = simulate_reduced_order_model(rom_config)
rom_df = pd.DataFrame(rom_result['trajectory'])
rom_df['year'] = 2026 + rom_df['step']

fig, ax = plt.subplots(figsize=(7.2, 3.8))
for i, var in enumerate(rom_config['state_variables']):
    ax.plot(rom_df['year'], rom_df[var], marker='o', linestyle=LINE_STYLES[i % len(LINE_STYLES)], label=var)
ax.set_xlabel('Year')
ax.set_ylabel('Reduced-order state value')
ax.set_title('Figure 13. Reduced-order system stress trajectory')
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
save_publication_figure(fig, 'fig_13_reduced_order_stress_trajectory', 'Reduced-order system stress trajectory', 'A transparent reduced-order abstraction shows how power, heat, water, yield, and legitimacy stress states evolve under a declared control policy.', 'ROM state variables', 'rom_df', 'Declared ROM configuration', 'strategic_simulation.simulate_reduced_order_model() reduced-order abstraction.', 'Not a high-fidelity fab physics simulator.')
plt.show()
display(pd.DataFrame([rom_result['diagnostics']]))

## 12. Validation readiness, evidence status, and unsupported claims

In [ ]:
reference_ranges = json.loads((ROOT / 'validation_lab/reference_ranges.json').read_text())
validation_rows, unknown_rows = [], []
for label, result in results_by_label.items():
    structure = validate_result_structure(result)
    ranges = validate_reference_ranges(result['summary'], reference_ranges)
    score = validation_scorecard([structure, ranges])
    validation_rows.append({
        'scenario_label': label,
        'scenario_id': result['metadata']['scenario_id'],
        'structural_passed': structure['passed'],
        'reference_ranges_passed': ranges['passed'],
        'reference_range_flags': len(ranges['range_flags']),
        **score,
    })
    unknown_rows.append({
        'scenario_label': label,
        'unknown_count': len(result['unknowns']['unresolved_variables']),
        'underdetermined': result['unknowns']['underdetermined'],
    })
validation_readiness_df = pd.DataFrame(validation_rows)
unknowns_df = pd.DataFrame(unknown_rows)
unsupported_claims_df = pd.DataFrame({'unsupported_claim': [
    'Verified Terafab operating data', 'Official Terafab endorsement or affiliation', 'Actual U.S. national energy or water impact as fact',
    'Permitting approval or environmental compliance', 'Investment merit or financing forecast', 'Construction, operation, adoption, or procurement proof',
    'External empirical validation beyond structural and public-reference screening'
]})
highest_value_next_data_df = pd.DataFrame({'next_data_need': [
    'Site-specific power capacity and tariff structure', 'Cooling design basis and heat-rejection system data', 'Permitted water withdrawal and discharge values',
    'Water-reuse and wastewater-treatment design assumptions', 'Yield, packaging, and qualification calibration data', 'External benchmark or reference dataset for validation'
]})
display(validation_readiness_df)
display(unknowns_df)
display(unsupported_claims_df)
display(highest_value_next_data_df)

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.1))
epivot = evidence_df.pivot_table(index='scenario_label', columns='evidence_status', values='count', aggfunc='sum', fill_value=0)
epivot.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab20')
axes[0].set_title('Evidence status'); axes[0].set_ylabel('Count'); axes[0].legend(frameon=False, fontsize=5)
validation_readiness_df.set_index('scenario_label')['score'].plot(kind='bar', ax=axes[1], color='#009E73')
axes[1].set_title('Validation score'); axes[1].set_ylim(0, 1.05)
unknowns_df.set_index('scenario_label')['unknown_count'].plot(kind='bar', ax=axes[2], color='#D55E00')
axes[2].set_title('Unresolved variables'); axes[2].set_ylabel('Count')
for ax in axes:
    ax.tick_params(axis='x', labelrotation=60)
fig.suptitle('Figure 14. Validation-readiness and evidence boundary', y=1.03)
fig.tight_layout()
save_publication_figure(fig, 'fig_14_validation_evidence_boundary', 'Validation-readiness and evidence boundary', 'Evidence composition, validation-readiness scores, and unresolved-variable counts define the claim boundary for the notebook.', 'evidence status; validation score; unknown count', 'evidence_df; validation_readiness_df; unknowns_df', ', '.join(summary_df['scenario_label'].unique()), 'validation_lab structural and public-reference screening.', 'Not external validation, certification, permitting approval, or verified Terafab data.')
plt.show()

## 13. Manuscript-ready table exports and figure manifest

In [ ]:
# Export manuscript-ready CSV tables.
scenario_registry_df.to_csv(FIG_DIR / 'table_01_scenario_registry.csv', index=False)
summary_df.to_csv(FIG_DIR / 'table_02_deterministic_summary.csv', index=False)
derived_summary_df.to_csv(FIG_DIR / 'table_03_derived_metrics.csv', index=False)
gates_df.to_csv(FIG_DIR / 'table_04_gate_matrix.csv', index=False)
monte_carlo_quantiles_df.to_csv(FIG_DIR / 'table_05_monte_carlo_quantiles.csv', index=False)
monte_carlo_sensitivity_df.to_csv(FIG_DIR / 'table_06_sensitivity_ranking.csv', index=False)
validation_readiness_df.to_csv(FIG_DIR / 'table_07_validation_readiness.csv', index=False)

figure_manifest_df = pd.DataFrame(figure_manifest)
figure_manifest_df.to_csv(FIG_DIR / 'table_08_figure_manifest.csv', index=False)
display(figure_manifest_df)
print(f'Exported {len(figure_manifest_df)} figures and manuscript tables to {FIG_DIR.relative_to(ROOT)}')

## 14. Conditional findings, limitations, and next data needs

**Main conditional findings.** The deterministic and uncertainty analyses quantify how declared Terafab-scale load assumptions propagate into modeled heat rejection, entropy generation, exergy destruction, water use, wastewater production, feasibility margins, and cost intensity through 2030.

**Assumption boundary.** Scenario inputs remain assumptions unless independently verified. Stress-test assumptions remain stress tests.

**Validation boundary.** The validation-readiness section performs structural and public-reference screening only. It is not external validation, certification, or site-specific engineering review.

**Unsupported claims.** This notebook does not establish verified Terafab operating data, official Terafab endorsement, actual U.S. national impacts as fact, permitting sufficiency, investment merit, construction, financing, operation, adoption, procurement, or empirical validation beyond public-reference screening.

**Highest-value next data.** Site-specific power capacity, cooling design basis, permitted withdrawal/discharge, water-reuse design, yield/readiness calibration, and external benchmark datasets would most improve publication-grade confidence.